## Review Classification Pipeline

==========================================

Zero-shot sentiment classification using a pretrained transformer,
evaluated against the star-rating-derived ground truth labels.


Model: cardiffnlp/twitter-roberta-base-sentiment-latest
  - Chosen because it natively outputs 3 classes (negative/neutral/positive),
    matching our 1-2/3/4-5 star mapping exactly. (distilbert-sst2 is binary
    only, so it doesn't fit our label scheme without extra work.)

Run:
    pip install transformers torch scikit-learn pandas tqdm
    python classification_pipeline.py --input merged_reviews_labeled.csv

If you're on CPU only and 65k rows is too slow for iteration, use
-- sample 5000 first to sanity-check the pipeline, then run the full set.

In [ ]:

import argparse
import re

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from tqdm import tqdm
from transformers import pipeline


# ---------------------------------------------------------------------------
# 1. Text cleaning
# ---------------------------------------------------------------------------
def clean_text(text: str) -> str:
    """Light cleaning — the model handles punctuation/case fine on its own,
    so we only strip things that add noise (urls, extra whitespace)."""
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ---------------------------------------------------------------------------
# 2. Load data
# ---------------------------------------------------------------------------
def load_data(path: str, sample: int | None = None) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = df.dropna(subset=["reviews.text", "sentiment"])
    df["clean_text"] = df["reviews.text"].apply(clean_text)
    df = df[df["clean_text"].str.len() > 0]
    if sample:
        df = df.sample(n=min(sample, len(df)), random_state=42)
    return df.reset_index(drop=True)


# ---------------------------------------------------------------------------
# 3. Run inference
# ---------------------------------------------------------------------------
# The model's raw output labels vary by checkpoint; this one uses
# 'negative' / 'neutral' / 'positive' directly, so no remapping needed.
# If you swap models, add a LABEL_MAP dict here to translate output -> our schema.
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
MAX_LEN = 512  # transformer input truncation limit


def run_classifier(texts: list[str], batch_size: int = 32) -> list[str]:
    classifier = pipeline(
        "sentiment-analysis",
        model=MODEL_NAME,
        truncation=True,
        max_length=MAX_LEN,
        device=-1,  # set to 0 if you have a GPU available
    )

    predictions = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Classifying"):
        batch = texts[i : i + batch_size]
        results = classifier(batch)
        predictions.extend([r["label"].lower() for r in results])
    return predictions


# ---------------------------------------------------------------------------
# 4. Evaluate
# ---------------------------------------------------------------------------
def evaluate(y_true: list[str], y_pred: list[str]) -> None:
    labels = ["negative", "neutral", "positive"]

    print("\n=== Accuracy ===")
    print(f"{accuracy_score(y_true, y_pred):.4f}")

    # Given the ~92/4/4 class imbalance, macro-F1 is the metric that
    # actually reflects performance on the minority classes — accuracy
    # alone would look artificially high even from a lazy model.
    print("\n=== Macro F1 (the metric that matters here, given imbalance) ===")
    print(f"{f1_score(y_true, y_pred, labels=labels, average='macro'):.4f}")

    print("\n=== Full classification report ===")
    print(classification_report(y_true, y_pred, labels=labels, digits=3))

    print("\n=== Confusion matrix (rows=true, cols=predicted) ===")
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(pd.DataFrame(cm, index=labels, columns=labels))


# ---------------------------------------------------------------------------
# 5. Main
# ---------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", default="merged_reviews_labeled.csv")
    parser.add_argument("--output", default="classified_reviews.csv")
    parser.add_argument(
        "--sample",
        type=int,
        default=None,
        help="Optional: sample N rows for a fast first run before doing the full dataset",
    )
    parser.add_argument("--batch_size", type=int, default=32)
    args = parser.parse_args()

    print(f"Loading data from {args.input} ...")
    df = load_data(args.input, sample=args.sample)
    print(f"Loaded {len(df)} rows")

    print(f"\nRunning {MODEL_NAME} ...")
    df["predicted_sentiment"] = run_classifier(
        df["clean_text"].tolist(), batch_size=args.batch_size
    )

    evaluate(df["sentiment"].tolist(), df["predicted_sentiment"].tolist())

    # This is the file Person B needs: real predicted sentiment per review,
    # to feed into their summarization / article-generation step.
    df.to_csv(args.output, index=False)
    print(f"\nSaved predictions to {args.output}")


if __name__ == "__main__":
    main()